In [1]:
import torch
import gc
import wandb
import warnings
import optuna

import torch.nn.functional as F
import torch.nn as nn
import numpy as np

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import ndcg_score
from torch.optim.lr_scheduler import StepLR

C:\Users\roans\anaconda3.0\envs\transformers\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
class TextInteractionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        print("Pre-tokenizing dataset (this takes a minute, but saves hours)...")
        self.labels = torch.tensor(dataframe['response'].values, dtype=torch.float)
        
        # Tokenize everything once and store in memory
        self.cv_encodings = tokenizer(
            dataframe['cv_text'].tolist(), add_special_tokens=True, 
            max_length=max_length, padding='max_length', 
            truncation=True, return_tensors='pt'
        )
        
        self.vac_encodings = tokenizer(
            dataframe['vacancy_text'].tolist(), add_special_tokens=True, 
            max_length=max_length, padding='max_length', 
            truncation=True, return_tensors='pt'
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'cv_input_ids': self.cv_encodings['input_ids'][idx],
            'cv_attention_mask': self.cv_encodings['attention_mask'][idx],
            'vac_input_ids': self.vac_encodings['input_ids'][idx],
            'vac_attention_mask': self.vac_encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

In [3]:
trainloader = torch.load(f'../dataloaders/text_trainloader.pth',
                         weights_only=False)
valloader = torch.load(f'../dataloaders/text_valloader.pth',
                         weights_only=False)
testloader = torch.load(f'../dataloaders/text_testloader.pth',
                         weights_only=False)

In [6]:
len(testloader.dataset)

644

In [4]:
def listwise_loss(scores, labels):
    """
    Compute the LambdaRank loss. (assume sigma=1.)
    """
    if labels.size(0) < 2:
        return torch.zeros_like(scores)

    N = torch.arange(len(scores))
    num_docs = len(scores)
    sigma = 1

    # Calculate lambda_{i, j} for every <i, j>.
    S_j = torch.stack([labels] * num_docs)
    S_i = S_j.T

    S = torch.nan_to_num((S_i - S_j) / (S_i - S_j).abs())
    lamda = (sigma * (0.5 * (1 - S) - (1 / (1 + torch.exp(sigma * (scores - scores.T))))))

    # Calculate abs(Delta-NDCG) for each ordering <i, j> combination
    sorted_ind = torch.flip(scores.argsort(dim=0).flatten(), dims=[0])
    sorted_labels = labels[sorted_ind]
    ideal_labels = torch.sort(labels)[0].flip(dims=[0])
    k = (torch.arange(sorted_labels.shape[0]) + 1).to(scores.device)
    
    DCG_ideal_labels = torch.sum((2**ideal_labels - 1) / torch.log(k + 1)) 
    
    # --- THE FIX: Prevent Division by Zero ---
    # If all labels are 0, ideal DCG is 0. There's no ranking to learn.
    if DCG_ideal_labels == 0:
        return torch.zeros_like(scores)
    # -----------------------------------------

    doc_id_to_rank = torch.Tensor([(sorted_ind == i).nonzero(as_tuple=True)[0] for i in N]).int()
    doc_id_to_label = torch.Tensor([sorted_labels[R_i] for R_i in doc_id_to_rank]).int().to(scores.device)
    
    # Calculate delta NDCG
    R_j = torch.stack([doc_id_to_rank] * num_docs).to(scores.device)
    R_i = R_j.T
    label_j = torch.stack([doc_id_to_label] * num_docs).to(scores.device)
    label_i = label_j.T
    DCG_discount = ((2**label_i - 1) / torch.log(R_i + 2) + (2**label_j - 1) / torch.log(R_j + 2)).to(scores.device)
    DCG_gain = ((2**label_j - 1) / torch.log(R_i + 2) + (2**label_i - 1) / torch.log(R_j + 2)).to(scores.device)
    delta_NDCG = ((DCG_gain - DCG_discount) / DCG_ideal_labels).abs()

    lambda_rank_loss =  (lamda * delta_NDCG).sum(axis=1).unsqueeze(1) 

    return lambda_rank_loss

In [5]:
class text_ranker(torch.nn.Module):
    def __init__(self, pooling="mean"):
        super().__init__()
        self.model = AutoModel.from_pretrained("jjzha/dajobbert-base-uncased")
        self.pooling = pooling
        
        for name, param in self.model.named_parameters():
            if 'encoder.layer' in name:
                layer_num = int(name.split('.')[2])
                if layer_num < 8:  # Freezes layers 0 through 7
                    param.requires_grad = False

    def pool_embeddings(self, outputs, attention_mask):
        """Helper function to cleanly pool the token embeddings"""
        if self.pooling == "mean":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size())
            sum_embeddings = torch.sum(outputs.last_hidden_state * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9) 
            return sum_embeddings / sum_mask
        elif self.pooling == "sum":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size())
            return torch.sum(outputs.last_hidden_state * input_mask_expanded, 1)
        elif self.pooling == "max":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).bool()
            masked_embeddings = outputs.last_hidden_state * input_mask_expanded 
            embeddings, _ = torch.max(masked_embeddings, dim=1) 
            return embeddings

    def forward(self, batch_cv, batch_vac):
        # 1. Embed the CV
        cv_outputs = self.model(batch_cv['input_ids'], attention_mask=batch_cv['attention_mask'])
        cv_emb = self.pool_embeddings(cv_outputs, batch_cv['attention_mask'])
        
        # 2. Embed the Vacancy
        vac_outputs = self.model(batch_vac['input_ids'], attention_mask=batch_vac['attention_mask'])
        vac_emb = self.pool_embeddings(vac_outputs, batch_vac['attention_mask'])
        
        # 3. Calculate Cosine Similarity 
        # F.cosine_similarity returns [batch_size], we unsqueeze to [batch_size, 1] for listwise_loss
        scores = F.cosine_similarity(cv_emb, vac_emb)
        return scores.unsqueeze(1)

In [10]:
def train_loop(model, optimizer, trainloader, use_wandb=False):
    ndcg_scores = []
    scaler = torch.cuda.amp.GradScaler() 
        
    for i, batch in enumerate(trainloader):
        batch_cv = {
            'input_ids': batch['cv_input_ids'].to(device),
            'attention_mask': batch['cv_attention_mask'].to(device)
        }
        batch_vac = {
            'input_ids': batch['vac_input_ids'].to(device),
            'attention_mask': batch['vac_attention_mask'].to(device)
        }
        ground_truth = batch['labels'].to(device)
        optimizer.zero_grad()

        # 1. Forward Pass in Mixed Precision
        with torch.cuda.amp.autocast():
            y_pred = model(batch_cv, batch_vac)
            if len(y_pred) > len(ground_truth):
                y_pred = y_pred[:len(ground_truth)]
                
            # listwise_loss calculates the EXPLICIT gradients, not a scalar loss
            lambda_i = listwise_loss(y_pred, ground_truth)
        
        # 2. Backward Pass
        scaled_lambda_i = scaler.scale(lambda_i)
        
        # Apply the scaled custom gradients backward through the network
        torch.autograd.backward(y_pred, scaled_lambda_i)
        
        # 3. Optimizer Step
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        
        scaler.step(optimizer)
        scaler.update()

        score = ndcg_score(ground_truth.detach().cpu().unsqueeze(0), 
                           y_pred.squeeze().unsqueeze(0).detach().cpu(), k=10)
        ndcg_scores.append(score)
        
        # flush=True prevents Jupyter from indefinitely buffering the output
        print(f"Batch: {i + 1}/{len(trainloader)}, y_pred mean: {y_pred.mean():.4f}, nDCG: {score:.4f}    ", end="\r", flush=True)
        
    return ndcg_scores


def val_loop(model, valloader):
    ndcg_scores = []
    
    with torch.no_grad():
        for i, batch in enumerate(valloader):
            print(f"Batch: {i + 1}/{len(valloader)}", end="\r")
            
            batch_cv = {
                'input_ids': batch['cv_input_ids'].to(device),
                'attention_mask': batch['cv_attention_mask'].to(device)
            }
            batch_vac = {
                'input_ids': batch['vac_input_ids'].to(device),
                'attention_mask': batch['vac_attention_mask'].to(device)
            }
            ground_truth = batch['labels'].to(device)
            
            y_pred_val = model(batch_cv, batch_vac)
                
            if len(y_pred_val) > len(ground_truth):
                y_pred_val = y_pred_val[:len(ground_truth)]
        
            score = ndcg_score(ground_truth.detach().cpu().unsqueeze(0), 
                               y_pred_val.squeeze().unsqueeze(0).detach().cpu(), k=10)
            ndcg_scores.append(score)
            
    return ndcg_scores

In [11]:
def train_model(trial, trainloader, valloader, epochs=10, step_size=5):
    best_score = 0
    
    # Search space
    learning_rate = trial.suggest_float('learning_rate', 1e-6, 1e-4, log=True)
    pooling_method = trial.suggest_categorical('pooling_method', ["mean", "max", "sum"])

    model = text_ranker(pooling=pooling_method).to(device)      
    model.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = StepLR(optimizer, step_size=3, gamma=0.1)
    
    for epoch in range(epochs + 1):
        print(f"Epoch: {epoch}/{epochs}")

        # Train the model for the current epoch
        epoch_ndcg_scores = train_loop(model, optimizer, trainloader)

        print(f"\nTraining nDCG: {np.mean(epoch_ndcg_scores):.4f}\n")
        scheduler.step()

        if use_wandb:
            wandb.log({"Training nDCG": np.mean(epoch_ndcg_scores)})

        # Evaluate the model
        val_ndcg_scores = val_loop(model, valloader)
        print(f"\nTesting nDCG: {np.mean(val_ndcg_scores):.4f}\n")

        if np.mean(val_ndcg_scores) > best_score:
            best_score = np.mean(val_ndcg_scores)
            
    return best_score

In [12]:
def objective_wrapper(trainloader, valloader):
    def objective(trial):
        # Make sure this calls train_model, NOT train_loop!
        return train_model(trial, trainloader, valloader) 
    
    return objective

In [13]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
use_wandb = False # Define this so it doesn't crash during the train loop check

torch.cuda.empty_cache() 
gc.collect()

# Hide user/future warnings
warnings.filterwarnings('ignore')

# Define the Optuna study
study = optuna.create_study(direction='maximize')

# We need to provide trainloader and valloader to the training/validation loop
wrapped_objective = objective_wrapper(trainloader, valloader)

# Start optimization
study.optimize(wrapped_objective, n_trials=5)  

print("Best hyperparameters:", study.best_trial.params)

[I 2026-06-24 16:17:07,711] A new study created in memory with name: no-name-0132fca4-e4e8-4c83-bb93-d442969dc5a5


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jjzha/dajobbert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch: 0/10
Batch: 337/337, y_pred mean: 0.6481, nDCG: 0.2891    
Training nDCG: 0.4635

Batch: 49/49
Testing nDCG: 0.4253

Epoch: 1/10
Batch: 337/337, y_pred mean: 0.1266, nDCG: 0.0000    
Training nDCG: 0.5191

Batch: 49/49
Testing nDCG: 0.4746

Epoch: 2/10
Batch: 337/337, y_pred mean: -0.0795, nDCG: 0.7094    
Training nDCG: 0.5373

Batch: 49/49
Testing nDCG: 0.4956

Epoch: 3/10
Batch: 337/337, y_pred mean: -0.2003, nDCG: 0.6257    
Training nDCG: 0.5286

Batch: 49/49
Testing nDCG: 0.4734

Epoch: 4/10
Batch: 337/337, y_pred mean: -0.2139, nDCG: 0.3010    
Training nDCG: 0.5406

Batch: 49/49
Testing nDCG: 0.4702

Epoch: 5/10
Batch: 337/337, y_pred mean: -0.2330, nDCG: 0.7904    
Training nDCG: 0.5289

Batch: 49/49
Testing nDCG: 0.4572

Epoch: 6/10
Batch: 337/337, y_pred mean: -0.3384, nDCG: 0.4413    
Training nDCG: 0.5472

Batch: 49/49
Testing nDCG: 0.4348

Epoch: 7/10
Batch: 337/337, y_pred mean: -0.2812, nDCG: 0.4218    
Training nDCG: 0.5386

Batch: 49/49
Testing nDCG: 0.4390

Ep

[I 2026-06-24 16:34:22,111] Trial 0 finished with value: 0.4955585559584321 and parameters: {'learning_rate': 3.080657190733998e-06, 'pooling_method': 'sum'}. Best is trial 0 with value: 0.4955585559584321.


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jjzha/dajobbert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch: 0/10
Batch: 337/337, y_pred mean: 0.8203, nDCG: 0.4693    
Training nDCG: 0.4714

Batch: 49/49
Testing nDCG: 0.4075

Epoch: 1/10
Batch: 337/337, y_pred mean: 0.4694, nDCG: 0.7041    
Training nDCG: 0.5225

Batch: 49/49
Testing nDCG: 0.4010

Epoch: 2/10
Batch: 337/337, y_pred mean: 0.3498, nDCG: 0.1413    
Training nDCG: 0.5375

Batch: 49/49
Testing nDCG: 0.4214

Epoch: 3/10
Batch: 337/337, y_pred mean: 0.2537, nDCG: 0.9709    
Training nDCG: 0.5596

Batch: 49/49
Testing nDCG: 0.4095

Epoch: 4/10
Batch: 337/337, y_pred mean: 0.0766, nDCG: 0.0000    
Training nDCG: 0.5724

Batch: 49/49
Testing nDCG: 0.3839

Epoch: 5/10
Batch: 337/337, y_pred mean: 0.1404, nDCG: 1.0000    
Training nDCG: 0.5758

Batch: 49/49
Testing nDCG: 0.3997

Epoch: 6/10
Batch: 337/337, y_pred mean: 0.1539, nDCG: 0.5000    
Training nDCG: 0.5534

Batch: 49/49
Testing nDCG: 0.3846

Epoch: 7/10
Batch: 337/337, y_pred mean: 0.1379, nDCG: 0.0000    
Training nDCG: 0.5593

Batch: 49/49
Testing nDCG: 0.3799

Epoch: 8

[I 2026-06-24 17:56:13,297] Trial 4 finished with value: 0.42358372052516163 and parameters: {'learning_rate': 8.63670516156536e-06, 'pooling_method': 'max'}. Best is trial 3 with value: 0.5192525865437477.


Batch: 49/49
Testing nDCG: 0.3949

Best hyperparameters: {'learning_rate': 8.493296693542725e-05, 'pooling_method': 'mean'}
